<a href="https://colab.research.google.com/github/JosseBergUiB/GEOV181_2026/blob/main/GEOV181_iButton_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Libraries needed
import requests
import pandas as pd
from google.colab import files

# iButton data

Open your ibutton data in a program like excel and remove the data from the time in between when you turned on the iButton and acutally put it at its location. And the same from the time taking it from the measuring location to the classroom. These values are not the values from outside, but the temperatures from in class, your backpack and at home. Save you file again as a csv.

## Loadoad and prepare your iButton data

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))

df_iButton = pd.read_csv(filename,skiprows=15,header=None,
    names=["DateTime", "Unit", "Temp_whole", "Temp_decimal"],encoding="latin1")

df_iButton.head()

In [ ]:
# Convert to datetime
df_iButton["DateTime"] = pd.to_datetime(df_iButton["DateTime"],format="%d.%m.%y %H:%M:%S")


In [ ]:
# Temperature
df_iButton["Temp_decimal"] = df_iButton["Temp_decimal"].fillna(0)

df_iButton["Temperature"] = (df_iButton["Temp_whole"].astype(str) + "."
    + df_iButton["Temp_decimal"].astype(int).astype(str)).astype(float)

df_iButton["Temperature"] = pd.to_numeric(df_iButton["Temperature"])

df_iButton = df_iButton.drop(columns=["Temp_whole", "Temp_decimal"])


In [ ]:
df_iButton.head()

## Load and prepare MET-Norge data

This first part is copied from your notebook from last week.

To access data from MET Norway we use their Frost API data service.

You first need to create a user number for yourself using your email address in the box named 'Create a user' here: https://frost.met.no/howto.html. This generates a unqiue client ID that you can then use to access the data that you want.

You then add your client ID into the code block below and start running the code to collect the data.

In [ ]:
# Insert your own client ID here
client_id = '<INSERT CLIENT ID HERE>'

In [ ]:
# Define endpoint and parameters to collect air temperature data
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN50539',
    'elements': 'mean(air_temperature P1D)',
    'referencetime': '2023-01-01/2023-12-31',
}
# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id,''))
# Extract JSON data
json = r.json()

In [ ]:
# Check if the request worked, print out any errors
if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])

In [ ]:
#creating an empty list
records = []

# Loop through the data
for i in range(len(data)):
    source_id = data[i]['sourceId']  # Accessing sourceId directly
    reference_time = data[i]['referenceTime']  # Accessing referenceTime directly
    observation = data[i]['observations'][0]  # Get the first (and only) observation

    # Create a record with the necessary fields
    record = {
        'sourceId': source_id,
        'referenceTime': reference_time,
        'value': observation['value'],
        'unit': observation['unit'],
        'timeOffset': observation['timeOffset'],
        'timeResolution': observation['timeResolution'],
        'timeSeriesId': observation['timeSeriesId'],
        'performanceCategory': observation['performanceCategory'],
        'exposureCategory': observation['exposureCategory'],
        'qualityCode': observation['qualityCode'],
    }

    # Append the record to the records list
    records.append(record)

# Create the dataFrame
df_MET_Norge = pd.DataFrame(records)

In [ ]:
# This defines the columns that you want to keep
columns = ['sourceId','referenceTime','value','unit','timeOffset']
df_MET = df_MET_Norge[columns].copy()
# Convert the time value to something Python understands
df_MET['referenceTime'] = pd.to_datetime(df_MET['referenceTime'])

In [ ]:
df_MET.head()

## Assignment iButton

1. Make a plot showing your data (df_iButtion) and the MET Norge (df_MET). If you are unsure how to do it, look at the previous exercise, use AI (but make sure you understand what you do) or ask me.

#Questions about iButton

2. What is the maximum daily temperature for the measurement period?
3. What is the minimum daily temperature for the measurement period?
4. What is the mean daily temperature and how does this vary through the measurement period (what is the standard deviation of daily air temperatures)?
5. What is the daily range (maximum daily temperature minus minimum daily temperature) of air temperature values?

#Questions about MET Norge

6. What is the maximum daily temperature for the measurement period?
7. What is the minimum daily temperature for the measurement period?
8. What is the mean daily temperature and how does this vary through the measurement period (what is the standard deviation of daily air temperatures)?
9. What is the daily range (maximum daily temperature minus minimum daily temperature) of air temperature values?

# Compare

10. How does your data and the MET_Norge data differ and why might that be?

# Rain gauge

## Load and prepare the data

If you have the data in excel, run the block below. If you want to add it by hand in Python, use the other code block.

### Load data useing excel

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))

df_rain = pd.read_excel(filename)

df_rain.head()

If your rain values use a comma as a decimal seperater instead of a ., run the code block below.

In [ ]:
for col in df_rain.columns:
    if df_rain[col].dtype == "object":
        try:
            df_rain[col] = pd.to_numeric(
                df_rain[col].astype(str).str.replace(",", ".")
            )
        except ValueError:
            pass

In [ ]:
df_rain.head()

Make a datetime column. If you have a seperate column for date and time, run the code block below. If you don't manage or have a different set up, try it yourself, ask AI or ask me.

In [ ]:
# Combine day and time
df_rain["datetime"] = pd.to_datetime(
    df_rain["Day"] + " " + df_rain["Time"],
    format="%d-%m-%Y %H:%M"
)

### Load data by hand

Replace my example data with your own data.

In [ ]:
# Load data by hand

df_rain = pd.DataFrame({
    "day": ["12-09-2026", "13-09-2026"],
    "time": ["12:00", "13:00"],
    "rain_mm": [12, 16]
})


In [ ]:
# Combine day and time
df_rain["datetime"] = pd.to_datetime(
    df_rain["day"] + " " + df_rain["time"],
    format="%d-%m-%Y %H:%M"
)


In [ ]:
df_rain.head()

## Assignment Rain

Plot the rain data. Think carefully about what type of plot you use. If you are unsure how to plot, have a look at the assignment from last week.